# Gold Layer — agg_restaurant_ranking (design 1.5, split credit)

One row per restaurant (`restaurant_id`, current version for name / address). **Full overwrite** daily (`CREATE OR REPLACE`, atomic): the score is non-additive, so it is always recomputed. Run **after** `gold/checks` and `gold/agg_restaurant_daily_views`.

| Column | Definition |
| --- | --- |
| `video_count` | `SUM(mention_count)` |
| `credited_views` | `SUM(latest view_count × mention_weight)` (split credit: a "top 10 quán" video gives each 1/10 of its views) |
| `credited_views_30d` | `SUM(views_gained in the last 30 days × mention_weight)` |
| `credited_views_7d` / `credited_views_prev_7d` | `SUM(credited_views_gained)` of `agg_restaurant_daily_views` over the last 7 snapshot days / the 7 days before (added 2026-10-31) |
| `growth_pct_7d` | `(7d − prev 7d) ÷ prev 7d`; NULL when prev 7d < `TREND_MIN_PREV_VIEWS` (500) |
| `trending_rank_in_city` | rank by `credited_views_7d` within the city, only restaurants with ≥ `TREND_MIN_VIEWS_7D` (1,000) views in 7 days, else NULL |
| `new_videos_7d` | videos mentioning the restaurant that were uploaded and first seen in the last 7 days |
| `like_rate` | `SUM(likes × weight) ÷ SUM(views × weight)` (never an average of per-video rates) |
| `ranking_score` | `0.7 × credited_views_30d + 0.3 × like_rate`, each min-max scaled 0–1 within the city (first version, tune later) |
| `rank_in_city` | rank by `ranking_score` within the city |
| `channel_count` | distinct YouTube channels that mention the restaurant (one channel can be a paid ad; several independent channels are harder to fake) |
| `google_rating_adjusted` | Bayesian average: `(rating × reviews + C × city mean) ÷ (reviews + C)`, `C = RATING_PRIOR_REVIEWS`, so 5.0 from 3 reviews does not beat 4.5 from 3,000 |
| `legit_score` | 0–100, weighted mean of the parts below over the parts that have data (`LEGIT_WEIGHTS`) |
| `legit_label` | `Legit` = `legit_score ≥ LEGIT_MIN_SCORE` **and** enough evidence (≥ `LEGIT_MIN_REVIEWS` Google reviews or ≥ 2 channels); `Ít dữ liệu` = < `LEGIT_MIN_REVIEWS` reviews and 1 channel; else `Bình thường` |

**legit_score parts (each 0–1):** `rating` = `(google_rating_adjusted − 3.5) ÷ 1.5` clipped · `reviews` = `log(1 + reviews) ÷ log(1 + REVIEWS_REF)` clipped to 1 (fixed reference 2,000 reviews since 2026-11-01; before it was the city maximum, so one Google place with huge review counts pulled every other score down) · `channels` = `min(channel_count, 5) ÷ 5` · `like_rate` and `views_30d` = the min-max scaled values of `ranking_score`. YouTube has no share count, so shares are not used.

**No Google rating → `legit_score` NULL** (label `Ít dữ liệu`) since 2026-11-01: YouTube parts alone (like rate, views) do not say whether a restaurant is good.

**District (2026-11-01):** `district` = the video's district, normalized (`Q.3` / `quận 03` → `Quận 3`, `Q. Gò Vấp` → `Gò Vấp`, drop `huyện` / `thị trấn` / `TP` / `(cũ)`); when the video has none, the old (pre-2025) district of the Google ward from `src/reference/vn_new_ward_to_old_district.csv` (`district_source` = `video` / `ward` / NULL). `district_raw` keeps the video text.

**Comments (2026-11-01):** `comment_count`, `comments_analyzed`, `positive_comment_share`, `negative_comment_share` from `agg_restaurant_comments` (split credit by `mention_weight`). `thumbnail_url` = thumbnail of the most viewed review video (Google returns no photos for this key).

Descriptive columns (`cuisine`, `price_band`, `google_rating`, `latitude`, ...) are copied from the current `dim_restaurant` version so the dashboard reads one table.

"Last 30 days" is relative to the latest snapshot day, so a rerun gives the same result.


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

AGG = f"{GOLD}.agg_restaurant_ranking"
WINDOW_DAYS = 30
DAILY = f"{GOLD}.agg_restaurant_daily_views"  # written just before by gold/agg_restaurant_daily_views
TREND_DAYS = 7
TREND_MIN_PREV_VIEWS = 500    # growth_pct_7d is NULL below this (a jump from 10 to 200 views is not +1,900 %)
TREND_MIN_VIEWS_7D = 1000     # trending_rank_in_city only for restaurants with at least this many views in 7 days
W_VIEWS_30D, W_LIKE_RATE = 0.7, 0.3
# legit_score (first version, tune after looking at the top 10)
LEGIT_WEIGHTS = {"rating": 0.35, "reviews": 0.20, "channels": 0.20, "like_rate": 0.15, "views_30d": 0.10}
RATING_PRIOR_REVIEWS = 50   # C in the Bayesian average
LEGIT_MIN_SCORE = 55        # dev run 31/10: best = 66.9 (4.7★, 7,084 reviews); 70 gave no Legit at all
LEGIT_MIN_REVIEWS = 100
REVIEWS_REF = 2000          # reviews part = log(1 + reviews) / log(1 + REVIEWS_REF), clipped to 1
COMMENTS = f"{GOLD}.agg_restaurant_comments"  # written just before by gold/agg_restaurant_comments (may not exist yet)
W = LEGIT_WEIGHTS

print(f"Gold source: {GOLD}.fact_video_restaurant_mention, fact_video_daily_snapshot, dim_restaurant")
print(f"Gold target: {AGG}")

# --- Old (pre-2025) district from the new Google ward ---
# Since the 2025 reform Google writes only the new ward ("..., Tân Hưng, Hồ Chí Minh") and no district.
# src/reference/vn_new_ward_to_old_district.csv maps each new ward to the old district most of it came from
# (same lookup as src/notebooks/ask/ask_restaurants.py).
import os, re, unicodedata
import pandas as pd

def _strip_accents(text):
    text = unicodedata.normalize("NFD", text.replace("đ", "d").replace("Đ", "D"))
    return "".join(c for c in text if unicodedata.category(c) != "Mn").lower()

def _key(text):
    text = re.sub(r"^\s*(phường|phuong|xã|xa|p\.|đặc khu|dac khu)\s*", "", text.strip(), flags=re.I)
    return re.sub(r"[^a-z0-9]", "", _strip_accents(text))

ref_path = next(p for p in ["../../reference/vn_new_ward_to_old_district.csv",
                            "src/reference/vn_new_ward_to_old_district.csv"] if os.path.exists(p))
ward_ref = pd.read_csv(ref_path, dtype=str).fillna("")
province_keys = {k: re.sub(r"^(thanhpho|tinh)", "", k) for k in ward_ref.province_key.unique()}
lookup = {(r.province_key, r.ward_short_key): r for r in ward_ref.itertuples()}

def _old_district(address, city):
    blob = _key(f"{address or ''} {city or ''}")
    provinces = [k for k, short in province_keys.items() if short in blob] or list(province_keys)
    for part in reversed((address or "").split(",")[:-1]):   # last part = city
        for pk in provinces:
            hit = lookup.get((pk, _key(part)))
            if hit is not None:
                return hit.district_old or None, hit.districts_old or None, hit.ward or None
    return None, None, None

addr = spark.sql(f"SELECT restaurant_id, address, city FROM {GOLD}.dim_restaurant WHERE is_current AND restaurant_key <> -1").toPandas()
old = [(r.restaurant_id, *_old_district(r.address, r.city)) for r in addr.itertuples()]
spark.createDataFrame(old, "restaurant_id STRING, district_old STRING, districts_old STRING, ward_new STRING") \
     .createOrReplaceTempView("old_district")
print(f"Old district found from the ward for {sum(1 for o in old if o[1])} / {len(old)} current restaurants")

# district text -> one display value ("Q.3" / "quận 03" -> "Quận 3", "Q. Gò Vấp" / "Huyện Củ Chi" -> "Gò Vấp" / "Củ Chi")
def district_norm_sql(col):
    return f"""CASE WHEN {col} RLIKE '(?iu)^(q\\\\.?\\\\s*|quận\\\\s+)0*[0-9]+$'
         THEN concat('Quận ', CAST(regexp_extract({col}, '([0-9]+)', 1) AS INT))
         ELSE nullif(trim(regexp_replace(regexp_replace({col},
                '(?iu)^(q\\\\.\\\\s*|quận\\\\s+|huyện\\\\s+|thị trấn\\\\s+|tp\\\\.?\\\\s*|thành phố\\\\s+)', ''),
                '\\\\s*\\\\(cũ\\\\)$', '')), '') END"""

# --- Comment metrics (empty when gold/agg_restaurant_comments has not run yet) ---
if spark.catalog.tableExists(COMMENTS):
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW restaurant_comment_stats AS
        SELECT restaurant_id,
               CAST(COUNT(DISTINCT comment_id) AS INT)                                    AS comment_count,
               CAST(COUNT(DISTINCT CASE WHEN sentiment IS NOT NULL THEN comment_id END) AS INT) AS comments_analyzed,
               SUM(CASE WHEN about_food AND sentiment = 'positive' THEN mention_weight END)
                 / NULLIF(SUM(CASE WHEN about_food AND sentiment IS NOT NULL THEN mention_weight END), 0) AS positive_comment_share,
               SUM(CASE WHEN about_food AND sentiment = 'negative' THEN mention_weight END)
                 / NULLIF(SUM(CASE WHEN about_food AND sentiment IS NOT NULL THEN mention_weight END), 0) AS negative_comment_share
        FROM {COMMENTS} GROUP BY restaurant_id
    """)
else:
    spark.sql("""CREATE OR REPLACE TEMP VIEW restaurant_comment_stats AS
                 SELECT CAST(NULL AS STRING) AS restaurant_id, 0 AS comment_count, 0 AS comments_analyzed,
                        CAST(NULL AS DOUBLE) AS positive_comment_share, CAST(NULL AS DOUBLE) AS negative_comment_share WHERE false""")

# --- Thumbnail of the most viewed review video per restaurant ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_thumbnail AS
    SELECT restaurant_id, max_by(thumbnail_url, view_count) AS thumbnail_url
    FROM (
        SELECT r.restaurant_id, v.thumbnail_url, s.view_count
        FROM {GOLD}.fact_video_restaurant_mention m
        JOIN {GOLD}.dim_restaurant r ON r.restaurant_key = m.restaurant_key
        JOIN {GOLD}.dim_video v      ON v.video_key = m.video_key
        LEFT JOIN (SELECT video_key, max_by(view_count, snapshot_date_key) AS view_count
                   FROM {GOLD}.fact_video_daily_snapshot GROUP BY video_key) s ON s.video_key = m.video_key
        WHERE m.restaurant_key <> -1 AND v.thumbnail_url IS NOT NULL
    )
    GROUP BY restaurant_id
""")


In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {AGG}
    COMMENT 'Restaurant ranking (split credit) + legit_score, full overwrite daily. Non-additive: never sum ranking_score / legit_score.'
    AS
    WITH latest_day AS (
        SELECT MAX(snapshot_date_key) AS k,
               CAST(date_format(date_sub(to_date(CAST(MAX(snapshot_date_key) AS STRING), 'yyyyMMdd'), {WINDOW_DAYS - 1}), 'yyyyMMdd') AS INT) AS window_start
        FROM {GOLD}.fact_video_daily_snapshot
    ),
    video_latest AS (   -- semi-additive counts: take each video's last snapshot
        SELECT video_key, view_count, like_count
        FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY video_key ORDER BY snapshot_date_key DESC) AS rn
              FROM {GOLD}.fact_video_daily_snapshot)
        WHERE rn = 1
    ),
    video_30d AS (      -- additive gains: sum over the window
        SELECT f.video_key, SUM(COALESCE(f.views_gained, 0)) AS views_gained_30d
        FROM {GOLD}.fact_video_daily_snapshot f CROSS JOIN latest_day l
        WHERE f.snapshot_date_key BETWEEN l.window_start AND l.k
        GROUP BY f.video_key
    ),
    trend AS (          -- 7-day window vs the 7 days before, from the restaurant x day table
        SELECT t.restaurant_id,
               SUM(CASE WHEN t.snapshot_date_key BETWEEN l.trend_start AND l.k THEN t.credited_views_gained ELSE 0 END)         AS credited_views_7d,
               SUM(CASE WHEN t.snapshot_date_key BETWEEN l.prev_start AND l.prev_end THEN t.credited_views_gained ELSE 0 END)  AS credited_views_prev_7d,
               SUM(CASE WHEN t.snapshot_date_key BETWEEN l.trend_start AND l.k THEN t.new_video_count ELSE 0 END)              AS new_videos_7d
        FROM {DAILY} t
        CROSS JOIN (
            SELECT k,
                   CAST(date_format(date_sub(to_date(CAST(k AS STRING), 'yyyyMMdd'), {TREND_DAYS - 1}), 'yyyyMMdd') AS INT)     AS trend_start,
                   CAST(date_format(date_sub(to_date(CAST(k AS STRING), 'yyyyMMdd'), {2 * TREND_DAYS - 1}), 'yyyyMMdd') AS INT) AS prev_start,
                   CAST(date_format(date_sub(to_date(CAST(k AS STRING), 'yyyyMMdd'), {TREND_DAYS}), 'yyyyMMdd') AS INT)         AS prev_end
            FROM latest_day
        ) l
        GROUP BY t.restaurant_id
    ),
    credited AS (
        SELECT d.restaurant_id,
               SUM(m.mention_count)                                  AS video_count,
               COUNT(DISTINCT dv.channel_id)                         AS channel_count,
               SUM(COALESCE(vl.view_count, 0) * m.mention_weight)    AS credited_views,
               SUM(COALESCE(v30.views_gained_30d, 0) * m.mention_weight) AS credited_views_30d,
               SUM(COALESCE(vl.like_count, 0) * m.mention_weight)    AS credited_likes,
               SUM(CASE WHEN vl.like_count IS NOT NULL THEN COALESCE(vl.view_count, 0) * m.mention_weight END) AS credited_views_with_likes
        FROM {GOLD}.fact_video_restaurant_mention m
        JOIN {GOLD}.dim_restaurant d ON d.restaurant_key = m.restaurant_key
        LEFT JOIN {GOLD}.dim_video dv ON dv.video_key = m.video_key AND dv.video_key <> -1
        LEFT JOIN video_latest vl ON vl.video_key = m.video_key
        LEFT JOIN video_30d v30   ON v30.video_key = m.video_key
        WHERE m.restaurant_key <> -1
        GROUP BY d.restaurant_id
        UNION ALL
        -- Google-only restaurants (Google Maps search, no video yet): no YouTube numbers
        SELECT d.restaurant_id, 0, 0, 0D, 0D, 0D, CAST(NULL AS DOUBLE)
        FROM {GOLD}.dim_restaurant d
        WHERE d.is_current AND d.restaurant_key <> -1 AND d.has_youtube_review = false
    ),
    enriched AS (
        SELECT c.*, COALESCE(tr.credited_views_7d, 0) AS credited_views_7d,
               COALESCE(tr.credited_views_prev_7d, 0) AS credited_views_prev_7d,
               COALESCE(tr.new_videos_7d, 0) AS new_videos_7d,
               cur.restaurant_name, cur.address, cur.district AS district_raw, COALESCE(cur.city, 'Unknown') AS city,
               od.district_old, od.districts_old, od.ward_new, th.thumbnail_url,
               cs.comment_count, cs.comments_analyzed, cs.positive_comment_share, cs.negative_comment_share,
               cur.latitude, cur.longitude, cur.cuisine, cur.cuisine_detail, cur.price_band, cur.price_min_vnd,
               cur.price_max_vnd, cur.google_rating, cur.google_rating_count, cur.google_maps_uri, cur.photo_name,
               COALESCE(cur.has_youtube_review, true) AS has_youtube_review,
               c.credited_likes / NULLIF(c.credited_views_with_likes, 0) AS like_rate
        FROM credited c
        JOIN {GOLD}.dim_restaurant cur ON cur.restaurant_id = c.restaurant_id AND cur.is_current
        LEFT JOIN trend tr ON tr.restaurant_id = c.restaurant_id
        LEFT JOIN old_district od ON od.restaurant_id = c.restaurant_id
        LEFT JOIN restaurant_thumbnail th ON th.restaurant_id = c.restaurant_id
        LEFT JOIN restaurant_comment_stats cs ON cs.restaurant_id = c.restaurant_id
    ),
    scaled AS (
        SELECT *,
            COALESCE((credited_views_30d - MIN(credited_views_30d) OVER w)
                     / NULLIF(MAX(credited_views_30d) OVER w - MIN(credited_views_30d) OVER w, 0), 0) AS views_30d_scaled,
            COALESCE((like_rate - MIN(like_rate) OVER w)
                     / NULLIF(MAX(like_rate) OVER w - MIN(like_rate) OVER w, 0), 0) AS like_rate_scaled
        FROM enriched
        WINDOW w AS (PARTITION BY city)
    ),
    rated AS (
        SELECT *,
            (google_rating * google_rating_count + {RATING_PRIOR_REVIEWS} * AVG(google_rating) OVER w)
                / (google_rating_count + {RATING_PRIOR_REVIEWS})                         AS google_rating_adjusted,
            least(ln(1 + google_rating_count) / ln(1 + {REVIEWS_REF}), 1)                   AS reviews_part
        FROM scaled
        WINDOW w AS (PARTITION BY city)
    ),
    parts AS (
        SELECT *,
            least(greatest((google_rating_adjusted - 3.5) / 1.5, 0), 1) AS rating_part,
            least(channel_count, 5) / 5.0                               AS channels_part
        FROM rated
    ),
    legit AS (   -- weighted mean over the parts that have data (no Google match -> YouTube parts only).
                 -- Google-only restaurant (no YouTube review): every YouTube part counts as 0, so its
                 -- score is at most 55 (rating 0.35 + reviews 0.20) and it ranks below YouTube-checked
                 -- restaurants with the same Google numbers (datle 2026-10-31)
        SELECT *,
            100 * ( COALESCE({W['rating']} * rating_part, 0) + COALESCE({W['reviews']} * reviews_part, 0)
                  + {W['channels']} * channels_part
                  + CASE WHEN like_rate IS NOT NULL THEN {W['like_rate']} * like_rate_scaled ELSE 0 END
                  + {W['views_30d']} * views_30d_scaled )
                / NULLIF( CASE WHEN rating_part IS NOT NULL THEN {W['rating']} ELSE 0 END
                  + CASE WHEN reviews_part IS NOT NULL THEN {W['reviews']} ELSE 0 END
                  + {W['channels']}
                  + CASE WHEN like_rate IS NOT NULL OR NOT has_youtube_review THEN {W['like_rate']} ELSE 0 END
                  + {W['views_30d']}, 0)
              * CASE WHEN google_rating IS NULL THEN NULL ELSE 1 END AS legit_score   -- no Google rating -> NULL
        FROM parts
    )
    SELECT
        restaurant_id, restaurant_name, address,
        COALESCE({district_norm_sql('district_raw')}, {district_norm_sql('district_old')}) AS district,
        CASE WHEN {district_norm_sql('district_raw')} IS NOT NULL THEN 'video'
             WHEN district_old IS NOT NULL THEN 'ward' END AS district_source,
        district_raw, districts_old, ward_new, city, latitude, longitude, thumbnail_url,
        COALESCE(comment_count, 0) AS comment_count, COALESCE(comments_analyzed, 0) AS comments_analyzed,
        round(positive_comment_share, 4) AS positive_comment_share, round(negative_comment_share, 4) AS negative_comment_share,
        cuisine, cuisine_detail, price_band, price_min_vnd, price_max_vnd,
        google_rating, google_rating_count, round(google_rating_adjusted, 2) AS google_rating_adjusted,
        google_maps_uri, photo_name, has_youtube_review,
        CASE WHEN NOT has_youtube_review THEN 'Chỉ Google'
             WHEN google_rating IS NULL THEN 'Chỉ YouTube'
             ELSE 'YouTube + Google' END          AS data_source,
        CAST(video_count AS INT)                  AS video_count,
        CAST(channel_count AS INT)                AS channel_count,
        CAST(round(credited_views) AS BIGINT)     AS credited_views,
        CAST(round(credited_views_30d) AS BIGINT) AS credited_views_30d,
        like_rate,
        CAST(credited_views_7d AS BIGINT)         AS credited_views_7d,
        CAST(credited_views_prev_7d AS BIGINT)    AS credited_views_prev_7d,
        CASE WHEN credited_views_prev_7d >= {TREND_MIN_PREV_VIEWS}
             THEN round((credited_views_7d - credited_views_prev_7d) / credited_views_prev_7d, 4) END AS growth_pct_7d,
        CASE WHEN credited_views_7d >= {TREND_MIN_VIEWS_7D}
             THEN CAST(RANK() OVER (PARTITION BY city, credited_views_7d >= {TREND_MIN_VIEWS_7D}
                                    ORDER BY credited_views_7d DESC, credited_views DESC) AS INT) END AS trending_rank_in_city,
        CAST(new_videos_7d AS INT)                AS new_videos_7d,
        {W_VIEWS_30D} * views_30d_scaled + {W_LIKE_RATE} * like_rate_scaled AS ranking_score,
        CAST(RANK() OVER (PARTITION BY city ORDER BY {W_VIEWS_30D} * views_30d_scaled + {W_LIKE_RATE} * like_rate_scaled DESC,
                                                     credited_views DESC) AS INT) AS rank_in_city,
        round(legit_score, 1)                     AS legit_score,
        CASE
            WHEN legit_score IS NULL THEN 'Ít dữ liệu'
            WHEN legit_score >= {LEGIT_MIN_SCORE} AND has_youtube_review   -- Legit needs a YouTube review
             AND (COALESCE(google_rating_count, 0) >= {LEGIT_MIN_REVIEWS} OR channel_count >= 2) THEN 'Legit'
            WHEN COALESCE(google_rating_count, 0) < {LEGIT_MIN_REVIEWS} AND channel_count < 2 THEN 'Ít dữ liệu'
            ELSE 'Bình thường'
        END                                       AS legit_label,
        CAST(RANK() OVER (PARTITION BY city ORDER BY legit_score DESC NULLS LAST, credited_views DESC) AS INT) AS legit_rank_in_city,
        (SELECT k FROM latest_day)                AS as_of_date_key,
        current_timestamp()                       AS etl_date
    FROM legit
""")


In [ ]:
# --- Verify ---
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT restaurant_id) AS ids, COUNT(DISTINCT city) AS cities, MAX(as_of_date_key) AS as_of
    FROM {AGG}
""").collect()[0]
print(f"📊 {AGG}: {s['n']} restaurants ({s['ids']} distinct ids) in {s['cities']} city(ies), as of {s['as_of']} "
      + ("✅" if s["n"] == s["ids"] else "❌ duplicates"))
display(spark.sql(f"SELECT rank_in_city, restaurant_name, district, video_count, credited_views, credited_views_30d, round(like_rate, 4) AS like_rate, round(ranking_score, 3) AS score FROM {AGG} ORDER BY city, rank_in_city LIMIT 20"))
display(spark.sql(f"""
    SELECT legit_rank_in_city, restaurant_name, cuisine, price_band, google_rating, google_rating_count,
           google_rating_adjusted, channel_count, round(like_rate, 4) AS like_rate, legit_score, legit_label
    FROM {AGG} ORDER BY city, legit_rank_in_city LIMIT 20
"""))
display(spark.sql(f"""
    SELECT trending_rank_in_city, restaurant_name, district, credited_views_7d, credited_views_prev_7d,
           round(100 * growth_pct_7d, 1) AS growth_pct, new_videos_7d
    FROM {AGG} WHERE trending_rank_in_city IS NOT NULL ORDER BY city, trending_rank_in_city LIMIT 20
"""))
display(spark.sql(f"SELECT legit_label, data_source, COUNT(*) AS n FROM {AGG} GROUP BY 1, 2 ORDER BY 1, 2"))
print("\n--- agg_restaurant_ranking complete ---")
